# 🔮 Capstone C3 · Teach the robot to imagine: a world model in DINOv2 space

**Capstone · stage 3 of 4** &nbsp;|&nbsp; ⏱ 90–120 min &nbsp;|&nbsp; 🖥️ Colab **T4 GPU** recommended

A policy answers *"what should I do?"*. A **world model** answers *"if I do this, what will I see next?"*. With one, a robot can **try actions in its imagination** before committing.

Following the idea behind **DINO-WM** (NYU/Meta) and **V-JEPA 2-AC** (Meta), our world model never paints pixels. It predicts **future DINOv2 features**, the same 4×4 tokens from C2 (lab 06 and the JEPA lecture explain why).

```
DINOv2 tokens now (16 × 384) ─┐
                              ├─► transformer ─► DINOv2 tokens 8 steps later ─► reward head: "coverage?"
next 8 actions ───────────────┘            ▲ feed back in to imagine further
```

### What you'll do
1. Train the world model and a **reward head** that predicts T-coverage from features.
2. Measure imagination honestly: does it beat *"nothing changes"* (persistence)?
3. **See** what it imagines, by retrieving the real frames closest to its predictions.
4. Run the research experiment: **C1's policy proposes N action chunks, the world model picks one.** Does imagination-based selection beat the policy alone? How close does it get to a *perfect* verifier? When does it fool itself (lab 03)?

In [ ]:
#@title 🔧 Step 0 · Run this cell first (click ▶). It loads the tools for this lab. { display-mode: "form" }
import importlib.util, subprocess, sys, os
def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
if importlib.util.find_spec("gym_pusht") is None:
    _pip("gym-pusht==0.1.6", "pymunk>=6.6,<7")          # pymunk 7 removed an API gym-pusht needs
if importlib.util.find_spec("zarr") is None or not __import__("zarr").__version__.startswith("2."):
    _pip("zarr==2.18.7", "numcodecs==0.15.1")
if importlib.util.find_spec("imageio") is None:
    _pip("imageio")

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

# ---------------------------------------------------------------------------
# Guided-lab helpers. You never need to edit this cell.
#  * ___            : a blank for you to fill in
#  * check(name, x) : checks your answer; if it is blank or wrong, it explains
#                     and hands back a working version so the notebook keeps going
#  * quiz(id)       : a clickable multiple-choice question
#  * playground(...) : sliders that re-run a function when you let go
# ---------------------------------------------------------------------------
import inspect, html as _html
import numpy as np
from IPython.display import display, HTML
import os
try:
    import ipywidgets as widgets
    _WIDGETS = not os.environ.get("GUIDE_NO_WIDGETS")
except Exception:
    _WIDGETS = False

class BlankNotFilled(Exception):
    pass

class _Blank:
    """The ___ placeholder. Any maths with it stops with a friendly message."""
    __array_ufunc__ = None
    def _stop(self, *args, **kwargs):
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _stop
    __truediv__ = __rtruediv__ = __floordiv__ = __rfloordiv__ = _stop
    __pow__ = __rpow__ = __matmul__ = __rmatmul__ = __mod__ = __rmod__ = _stop
    __neg__ = __pos__ = __abs__ = __getitem__ = __call__ = __iter__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __bool__ = __float__ = __int__ = __index__ = _stop
    __array__ = __len__ = _stop
    def __getattr__(self, name):
        if name.startswith('__'):
            raise AttributeError(name)
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    def __repr__(self):
        return "___"

___ = _Blank()
CHALLENGES, QUIZZES = {}, {}
_solved, _quiz_score = {}, {}

_STYLE = {
    "ok":   ("#e8f6ee", "#1b7a4b", "✅"),
    "wait": ("#fff5e0", "#9a5b00", "🧩"),
    "bad":  ("#fdecea", "#b3261e", "❌"),
    "info": ("#eaf1fb", "#245eb5", "💡"),
}

def card(kind, title, body=""):
    bg, fg, icon = _STYLE[kind]
    display(HTML(
        f'<div style="background:{bg};border-left:5px solid {fg};padding:10px 14px;'
        f'border-radius:6px;margin:6px 0;color:#1d2530;font-size:14px;line-height:1.5">'
        f'<b style="color:{fg}">{icon} {title}</b><div>{body}</div></div>'))

def _as_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        return [_as_numpy(v) for v in x]
    return x

def _same(a, b, tol):
    a, b = _as_numpy(a), _as_numpy(b)
    if isinstance(a, list) or isinstance(b, list):
        return isinstance(a, list) and isinstance(b, list) and len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    try:
        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    except Exception:
        return a == b
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def _has_blank(obj):
    if isinstance(obj, _Blank):
        return True
    if isinstance(obj, dict):
        return any(_has_blank(v) for v in obj.values())
    if isinstance(obj, (list, tuple)):
        return any(_has_blank(v) for v in obj)
    if callable(obj):
        try:
            return "___" in inspect.getsource(obj)
        except Exception:
            return False
    return False

def check(name, answer):
    """Check a challenge. Returns your answer if it works, otherwise a working reference."""
    ch = CHALLENGES[name]
    ref = ch["reference"]
    title = ch.get("title", name)
    fallback = ("<br><i>For now the notebook will use a working version so every later cell still runs. "
                "Come back, fill it in, and re-run this cell.</i>")
    if _has_blank(answer):
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” is waiting for you", "Hint: " + ch["hint"] + fallback)
        return ref
    try:
        if "test" in ch:
            ok, message = ch["test"](answer)
        elif callable(ref):
            ok, message = True, ""
            for args in ch["cases"]:
                args = args if isinstance(args, tuple) else (args,)
                expected, got = ref(*args), answer(*args)
                if not _same(expected, got, ch.get("tol", 1e-6)):
                    ok = False
                    message = "For a test input your function gave a different result from the expected one."
                    break
        else:
            ok = _same(ref, answer, ch.get("tol", 1e-6))
            message = f"You entered <code>{_html.escape(repr(_as_numpy(answer)))}</code>."
    except BlankNotFilled:
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” still has a blank", "Hint: " + ch["hint"] + fallback)
        return ref
    except Exception as err:
        ok, message = False, f"Running your version raised <code>{_html.escape(type(err).__name__)}: {_html.escape(str(err))}</code>."
    if ok:
        _solved[name] = True
        card("ok", f"Challenge solved: {title}", ch.get("why", ""))
        return answer
    _solved[name] = False
    card("bad", f"Not quite yet: {title}", message + "<br>Hint: " + ch["hint"] + fallback)
    return ref

def quiz(qid):
    q = QUIZZES[qid]
    question = f'<div style="font-size:15px;margin:8px 0 4px"><b>{"🔮 Predict: " if q.get("predict") else "🤔 "}{q["q"]}</b></div>'
    if not _WIDGETS:
        options = "".join(f"<li>{_html.escape(o)}</li>" for o in q["options"])
        display(HTML(question + f"<ol type='A'>{options}</ol><details><summary>Answer</summary>"
                     f"{'ABCDEFG'[q['answer']]}. {q['explain']}</details>"))
        return
    out = widgets.Output()
    buttons = []
    def choose(i):
        def handler(_):
            _quiz_score.setdefault(qid, i == q["answer"])
            for j, b in enumerate(buttons):
                b.button_style = "success" if j == q["answer"] else ("danger" if j == i else "")
            with out:
                out.clear_output()
                if i == q["answer"]:
                    card("ok", "Yes!", q["explain"])
                else:
                    card("bad", "Not this one. Here is the reasoning:", q["explain"])
        return handler
    for i, option in enumerate(q["options"]):
        b = widgets.Button(description=f"{'ABCDEFG'[i]}. {option}", layout=widgets.Layout(width="auto", max_width="100%"))
        b.on_click(choose(i))
        buttons.append(b)
    display(HTML(question), widgets.VBox(buttons), out)

def playground(fn, **controls):
    """controls: name=(min, max, step, default) for sliders, or name=[option, ...] for a dropdown."""
    defaults, sliders = {}, {}
    for name, spec in controls.items():
        if isinstance(spec, list):
            defaults[name] = spec[0]
            if _WIDGETS:
                sliders[name] = widgets.Dropdown(options=spec, value=spec[0], description=name)
        else:
            lo, hi, step, value = spec
            defaults[name] = value
            if _WIDGETS:
                kind = widgets.IntSlider if all(isinstance(v, int) for v in spec) else widgets.FloatSlider
                sliders[name] = kind(min=lo, max=hi, step=step, value=value, description=name,
                                     continuous_update=False, style={"description_width": "initial"},
                                     layout=widgets.Layout(width="420px"))
    if _WIDGETS:
        ui = widgets.VBox(list(sliders.values()))
        out = widgets.interactive_output(fn, sliders)
        display(ui, out)
    else:
        fn(**defaults)

def progress_report():
    solved = sum(_solved.values()); total = len(CHALLENGES)
    right = sum(_quiz_score.values()); asked = len(_quiz_score)
    stars = "⭐" * solved + "☆" * (total - solved)
    body = f"Challenges solved yourself: <b>{solved} / {total}</b> {stars}<br>"
    body += f"Quiz questions right on the first click: <b>{right} / {asked}</b> (of {len(QUIZZES)} in this lab)"
    missing = [CHALLENGES[k].get('title', k) for k in CHALLENGES if not _solved.get(k)]
    if missing:
        body += "<br>Still worth a try: " + ", ".join(missing)
    card("info", "Your progress in this lab", body)

# ---- this lab's challenges and quizzes ----
CHALLENGES["residual"] = dict(title="Predict the change, not the whole future",
    reference=lambda z, delta: z + delta,
    cases=[(torch.randn(2, 16, 384), torch.randn(2, 16, 384))],
    hint="Next features = current features + predicted change.",
    why="Most of the scene (walls, target, background) doesn't change in 0.8 s. Predicting a residual lets the network focus on what moves: lab 01's “predict the delta” at transformer scale.")

def _test_rollout(fn):
    class Toy(nn.Module):
        def forward(self, z, a): return z + a.sum((1, 2)).view(-1, 1, 1)
    z0 = torch.zeros(3, 16, 4); a = torch.ones(3, 2, 8, 2)
    out = fn(Toy(), z0, a)
    ok = isinstance(out, (list, tuple)) and len(out) == 2 and torch.allclose(out[1], torch.full((3, 16, 4), 32.0))
    return ok, "The second prediction should start from the FIRST PREDICTION, not from the real features."
def _ref_rollout(model, z0, action_blocks):
    preds, z = [], z0
    for k in range(action_blocks.shape[1]):
        z = model(z, action_blocks[:, k])
        preds.append(z)
    return preds
CHALLENGES["imagine"] = dict(title="Train on its own imagination", reference=_ref_rollout, test=_test_rollout,
    hint="Inside the loop, feed <code>z</code> (the model's latest prediction) back into the model: <code>z = model(z, action_blocks[:, k])</code>.",
    why="Training on 2-step imagined rollouts teaches the model to cope with its own small errors, a standard trick against compounding error (lab 01).")

CHALLENGES["nearest"] = dict(title="Find the closest real frame",
    reference=lambda query, bank: torch.cdist(query, bank).argmin(dim=1),
    cases=[(torch.tensor([[0., 0.], [5., 5.]]), torch.tensor([[4., 4.], [0.1, 0.], [9., 9.]]))],
    hint="<code>torch.cdist(query, bank)</code> gives every query-to-bank distance; take <code>argmin</code> along the bank dimension (dim=1).",
    why="A world model that predicts features can't directly show a picture. Retrieving the nearest real frame is a cheap, honest way to <i>visualise</i> imagination without training a decoder.")

CHALLENGES["pick"] = dict(title="Let imagination choose",
    reference=lambda predicted_coverage: int(torch.argmax(predicted_coverage)),
    cases=[torch.tensor([0.2, 0.9, 0.5]), torch.tensor([0.7, 0.1])],
    hint="Pick the index of the candidate whose imagined future has the highest predicted coverage.",
    why="This is <b>best-of-N with a learned verifier</b>, the same pattern as reward-model reranking for LLMs and value-guided sampling for robot policies.")

QUIZZES["persistence"] = dict(predict=True, q="Predicting DINOv2 features 8 steps ahead: how will a trained world model compare with “nothing changes” (persistence)?",
    options=["Worse: persistence is unbeatable", "Better, and the advantage grows at longer horizons as more of the scene moves", "Exactly the same"],
    answer=1, explain="At short horizons little changes, so persistence is decent. As the pusher and block move, copying the present gets worse, and a model that uses the actions pulls ahead. Always report this baseline (lab 04).")
QUIZZES["verifier"] = dict(predict=True, q="C1's policy proposes 32 action chunks and the world model picks the one with the best imagined coverage. Compared with using the policy's first proposal…",
    options=["it must be better: more options + a model", "it could be better or worse: the policy is already good, and a picky search can exploit world-model mistakes", "it must be worse"],
    answer=1, explain="Lab 03's optimiser's curse. In our test run, world-model picking with 32 candidates <b>hurt</b> (0.61 vs 0.80 for the policy alone), 8 candidates was roughly neutral, and a <b>perfect</b> verifier with the same 8 candidates reached 0.96. The candidates were good enough; the learned ranking was the bottleneck. That's why the random-pick control and the perfect upper bound matter.")
print('✅ Setup complete. Scroll down and run the cells in order.')

In [ ]:
#@title 🧰 Capstone toolkit · run me (data, simulator, evaluation, GIFs). Read the notes below; no need to edit. { display-mode: "form" }
import os, json, time, math, copy, hashlib, urllib.request
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import gymnasium as gym, gym_pusht, zarr, imageio
from IPython.display import Image as _GifImage
plt.rcParams.update({"figure.dpi": 110})

FAST_DEV_RUN = os.environ.get("CAPSTONE_FAST") == "1"      # course authors' quick self-test switch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
OUT = Path("capstone_outputs"); OUT.mkdir(exist_ok=True)
print("device:", DEVICE, "· outputs folder:", OUT.resolve())
if DEVICE.type == "cpu":
    print("⚠️  No GPU found. In Colab: Runtime → Change runtime type → T4 GPU. (CPU works, just slower.)")

# ---------------- data ----------------
DATA_URL = "https://diffusion-policy.cs.columbia.edu/data/training/pusht.zip"
def load_pusht(with_images=False):
    archive = Path("pusht.zip")
    if not archive.exists():
        print("Downloading PushT (31 MB)…"); urllib.request.urlretrieve(DATA_URL, archive)
    root = zarr.open_group(store=zarr.ZipStore(str(archive), mode="r"), mode="r", path="pusht/pusht_cchi_v7_replay.zarr")
    data = dict(state=root["data/state"][:].astype(np.float32), action=root["data/action"][:].astype(np.float32),
                ends=root["meta/episode_ends"][:])
    data["starts"] = np.r_[0, data["ends"][:-1]]
    if with_images:
        imgs = np.empty((len(data["state"]), 96, 96, 3), np.uint8)
        for i in range(0, len(imgs), 2048):
            imgs[i:i + 2048] = root["data/img"][i:i + 2048]
        data["images"] = imgs
    order = np.random.default_rng(42).permutation(len(data["ends"]))         # the SAME split as lab 04
    data["train_eps"], data["val_eps"], data["test_eps"] = order[:144], order[144:175], order[175:]
    return data

# ---------------- simulator ----------------
def make_env(obs_type="state"):
    return gym.make("gym_pusht/PushT-v0", obs_type=obs_type)

def sim_state(env):
    u = env.unwrapped
    return np.array([*u.agent.position, *u.block.position, u.block.angle % (2 * np.pi)], np.float32)

def set_dataset_state(env, s):
    """Put the simulator exactly into a recorded dataset state.
    gym-pusht's own reset_to_state sets the block angle AFTER its position. With modern pymunk that rotates
    the T around its centre of mass and moves it ~90 units away from where the dataset recorded it.
    Setting the angle FIRST reproduces the recorded frames pixel-for-pixel (we verified this on all 206 demos)."""
    u = env.unwrapped
    u.agent.position = list(map(float, s[:2])); u.agent.velocity = (0, 0)
    u.block.angle = float(s[4]); u.block.position = list(map(float, s[2:4]))
    u.block.velocity = (0, 0); u.block.angular_velocity = 0
    u.space.step(u.dt)
    return u.get_obs()

def coverage_of(env):
    return float(env.unwrapped._get_coverage())

def score_from_best_coverage(best):
    return min(best / 0.95, 1.0)

# ---------------- evaluation ----------------
EVAL_SEEDS = list(range(100000, 100050))       # 50 fixed start states, identical for every experiment

def run_episode(choose_chunk, seed, execute=8, obs_type="state", record=False, max_steps=300):
    """choose_chunk(history) -> array of future actions (world units). history = list of past observations."""
    env = make_env(obs_type)
    obs, _ = env.reset(seed=seed)
    history, best, frames, steps = [obs, obs], 0.0, [], 0
    while steps < max_steps:
        chunk = choose_chunk(history[-2:], env)
        for a in chunk[:execute]:
            obs, _, terminated, _, info = env.step(np.asarray(a, np.float64))
            history.append(obs); steps += 1
            best = max(best, info["coverage"])
            if record and steps % 2 == 0:
                frames.append(env.unwrapped.render()[::3, ::3])
            if terminated or steps >= max_steps:
                break
        if terminated:
            break
    env.close()
    return dict(best_coverage=best, score=score_from_best_coverage(best), success=best > 0.95, steps=steps, frames=frames)

def bootstrap_ci(values, repeats=2000, seed=0):
    values = np.asarray(values, float); r = np.random.default_rng(seed)
    means = values[r.integers(0, len(values), (repeats, len(values)))].mean(1)
    return float(values.mean()), float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))

def evaluate(choose_chunk, seeds=EVAL_SEEDS, execute=8, obs_type="state", label="policy", verbose=True):
    t0 = time.time(); rows = [run_episode(choose_chunk, s, execute, obs_type) for s in seeds]
    scores = [r["score"] for r in rows]
    mean, lo, hi = bootstrap_ci(scores)
    result = dict(label=label, score=mean, ci95=[lo, hi], success_rate=float(np.mean([r["success"] for r in rows])),
                  reached_80pct=float(np.mean([r["best_coverage"] > 0.8 for r in rows])), episodes=len(rows),
                  seconds=round(time.time() - t0, 1), per_episode_scores=scores)
    if verbose:
        print(f"{label}: score {mean:.3f} (95% CI {lo:.3f}–{hi:.3f}) · ≥80% coverage in {result['reached_80pct']:.0%} "
              f"· full success {result['success_rate']:.0%} · {len(rows)} episodes in {result['seconds']:.0f}s")
    return result

def save_gif(frames, name):
    path = OUT / name
    imageio.mimsave(path, frames, duration=0.1, loop=0)
    return path

def show_gif(path, width=280):
    display(_GifImage(filename=str(path), width=width))

def save_results(name, obj):
    (OUT / name).write_text(json.dumps(obj, indent=1))
    print("saved", OUT / name)

def human_reference(data, episodes):
    env = make_env("state"); env.reset(seed=0); best = []
    for e in episodes:
        b = 0.0
        for i in range(data["starts"][e], data["ends"][e]):
            set_dataset_state(env, data["state"][i]); b = max(b, coverage_of(env))
        best.append(score_from_best_coverage(b))
    return float(np.mean(best))


def to_unit(x):          # world units 0..512  ->  -1..1
    return x / 256.0 - 1.0
def from_unit(x):        # -1..1  ->  world units 0..512
    return (x + 1.0) * 256.0
def state_features(s):   # (…, 5) -> (…, 6): positions in -1..1, angle as sin & cos
    return np.concatenate([to_unit(s[..., :4]), np.sin(s[..., 4:5]), np.cos(s[..., 4:5])], -1).astype(np.float32)

class ResBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm = nn.LayerNorm(width)
        self.ff = nn.Sequential(nn.Linear(width, 4 * width), nn.SiLU(), nn.Linear(4 * width, width))
    def forward(self, h):
        return h + self.ff(self.norm(h))

class FlowPolicy(nn.Module):
    """Velocity network v(noisy action chunk, time, observation)."""
    def __init__(self, obs_dim=12, horizon=16, act_dim=2, width=512, depth=3):
        super().__init__()
        self.horizon, self.act_dim = horizon, act_dim
        self.inp = nn.Linear(horizon * act_dim + obs_dim + 32, width)
        self.blocks = nn.Sequential(*[ResBlock(width) for _ in range(depth)])
        self.out = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, horizon * act_dim))
        self.register_buffer("freqs", torch.exp(torch.linspace(0, math.log(1000), 16)))
    def forward(self, x_t, t, obs):
        tf = t * self.freqs
        h = self.inp(torch.cat([x_t.flatten(1), obs, torch.sin(tf), torch.cos(tf)], 1))
        return self.out(self.blocks(h)).view(-1, self.horizon, self.act_dim)


def load_or_train_c1_policy(data, steps=12000):
    net = FlowPolicy().to(DEVICE)
    ckpt = OUT / "c1_flow_policy.pt"
    if ckpt.exists():
        net.load_state_dict(torch.load(ckpt, map_location=DEVICE)); print("loaded your C1 policy from", ckpt)
        return net.eval()
    print("c1_flow_policy.pt not found in capstone_outputs/, so training one with C1's recipe (upload yours to skip this)…")
    S, A, st, en = data["state"], data["action"], data["starts"], data["ends"]
    obs, chunks = [], []
    for e in data["train_eps"]:
        for i in range(st[e], en[e]):
            obs.append(np.concatenate([state_features(S[max(i - 1, st[e])]), state_features(S[i])]))
            chunks.append(to_unit(A[np.minimum(np.arange(i, i + 16), en[e] - 1)]))
    O, C = torch.tensor(np.array(obs), device=DEVICE), torch.tensor(np.array(chunks), device=DEVICE)
    ema = copy.deepcopy(net).eval(); opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
    steps = 300 if FAST_DEV_RUN else steps
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=steps, pct_start=0.05)
    for step in range(steps):
        b = torch.randint(0, len(O), (256,), device=DEVICE); x1 = C[b]; x0 = torch.randn_like(x1); t = torch.rand(256, 1, 1, device=DEVICE)
        loss = F.mse_loss(net((1 - t) * x0 + t * x1, t.view(256, 1), O[b]), x1 - x0)
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step()
        with torch.no_grad():
            d = min(0.999, (1 + step) / (10 + step))
            for pe, pn in zip(ema.parameters(), net.parameters()):
                pe.mul_(d).add_(pn.detach(), alpha=1 - d)
    torch.save(ema.state_dict(), ckpt)
    return ema

@torch.no_grad()
def sample_state_chunks(net, prev_state, state, n=1, flow_steps=10, horizon=16):
    obs = torch.tensor(np.concatenate([state_features(prev_state), state_features(state)]), device=DEVICE)[None].repeat(n, 1)
    x = torch.randn(n, horizon, 2, device=DEVICE)
    for k in range(flow_steps):
        x = x + net(x, torch.full((n, 1), k / flow_steps, device=DEVICE), obs) / flow_steps
    return from_unit(x.clamp(-1, 1)).cpu().numpy()

---
## 1 · Ingredients: states, actions, features, and a reward label for every frame 🧂
C2 saved `capstone_outputs/dino_4x4.npy`. If it's missing (new Colab session?), upload it to `capstone_outputs/` or let the cell below re-extract (~5 min on a T4).

In [ ]:
data = load_pusht(with_images=True)
S, A_, IMGS, starts, ends = data["state"], data["action"], data["images"], data["starts"], data["ends"]

dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", trust_repo=True).eval().to(DEVICE)
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
@torch.no_grad()
def dino_tokens(images_uint8):                      # identical preprocessing to C2
    x = torch.as_tensor(np.asarray(images_uint8), device=DEVICE).permute(0, 3, 1, 2).float() / 255
    x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)
    tok = dino.forward_features((x - MEAN) / STD)["x_norm_patchtokens"]
    return F.adaptive_avg_pool2d(tok.transpose(1, 2).reshape(len(x), 384, 16, 16), 4).flatten(2).transpose(1, 2)

FEAT_FILE = OUT / "dino_4x4.npy"
if not FEAT_FILE.exists():
    print("No cached features found, extracting…"); feats = np.zeros((len(IMGS), 16, 384), np.float16)
    for i in range(0, len(IMGS), 128):
        feats[i:i + 128] = dino_tokens(IMGS[i:i + 128]).half().cpu().numpy()
    np.save(FEAT_FILE, feats)
FT = torch.from_numpy(np.load(FEAT_FILE).astype(np.float32))
train_frames = np.concatenate([np.arange(starts[e], ends[e]) for e in data["train_eps"]])
MU, SD = FT[train_frames].mean((0, 1)), FT[train_frames].std((0, 1)) + 1e-6
Z = ((FT - MU) / SD).to(DEVICE)
ACT = torch.from_numpy(to_unit(A_)).to(DEVICE)
print("features", tuple(Z.shape))

# Reward label for every frame: put the simulator into that recorded state and measure coverage.
env = make_env("state"); env.reset(seed=0); t0 = time.time()
COVERAGE = np.array([ (set_dataset_state(env, s), coverage_of(env))[1] for s in S ], np.float32)
COV = torch.from_numpy(COVERAGE).to(DEVICE)
print(f"coverage labels for {len(COVERAGE)} frames in {time.time() - t0:.0f}s · average {COVERAGE.mean():.2f}")

---
## 2 · The world model 🌍
* **Input:** 16 feature tokens (projected 384 → 192) plus **one action token** built from the next 8 actions.
* **Body:** a 3-layer transformer, so every token can attend to the action and to every other patch.
* **Output:** a *change* for each of the 16 tokens.
* **Reward head:** a small MLP that reads 16 tokens and predicts coverage (0–1).

### 🧩 Challenge 1 · Predict the change, not the whole future

In [ ]:
K = 8                                      # environment steps per imagined step (0.8 s)

def next_features(z, delta):
    return ___                     # 🧩 residual prediction

next_features = check("residual", next_features)

class WorldModel(nn.Module):
    def __init__(self, d=192, layers=3):
        super().__init__()
        self.inp = nn.Linear(384, d)
        self.pos = nn.Parameter(torch.zeros(1, 17, d))                    # 1 action token + 16 patch tokens
        self.act = nn.Sequential(nn.Linear(2 * K, d), nn.SiLU(), nn.Linear(d, d))
        block = nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=4 * d, dropout=0.0, batch_first=True, norm_first=True, activation="gelu")
        self.transformer = nn.TransformerEncoder(block, layers, enable_nested_tensor=False)
        self.out = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 384))
        self.reward = nn.Sequential(nn.Linear(16 * 384, 256), nn.SiLU(), nn.Linear(256, 1))
    def forward(self, z, actions):                                        # z (B,16,384), actions (B,K,2)
        h = torch.cat([self.act(actions.flatten(1))[:, None], self.inp(z)], 1) + self.pos
        return next_features(z, self.out(self.transformer(h))[:, 1:])
    def coverage(self, z):
        return torch.sigmoid(self.reward(z.flatten(1))).squeeze(1)

wm = WorldModel().to(DEVICE)
print(f"world model parameters: {sum(p.numel() for p in wm.parameters()) / 1e6:.1f} M")

<details><summary>🤔 <b>Need a hint?</b></summary>

Add the predicted change to the current features.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return z + delta                     # 🧩 residual prediction</pre>

</details>

### 🧩 Challenge 2 · Train on its own imagination

In [ ]:
def imagine(model, z0, action_blocks):          # action_blocks: (B, steps, K, 2)
    preds, z = [], z0
    for k in range(action_blocks.shape[1]):
        z = ___     # 🧩 feed the latest prediction back in
        preds.append(z)
    return preds

imagine = check("imagine", imagine)

def window_starts(episodes, n_blocks):
    return np.concatenate([np.arange(starts[e], ends[e] - n_blocks * K) for e in episodes if ends[e] - starts[e] > n_blocks * K])

<details><summary>🤔 <b>Need a hint?</b></summary>

Use <code>z</code>, not <code>z0</code>, inside the loop.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>z = model(z, action_blocks[:, k])     # 🧩 feed the latest prediction back in</pre>

</details>

## 3 · Train ⏳
Loss = feature error at imagined steps 1 **and** 2, plus reward-head error (binary cross-entropy vs true coverage).

⏱ 8,000 steps took about 5 minutes on our laptop GPU. The cell prints a live estimate.

In [ ]:
WM_STEPS = 400 if FAST_DEV_RUN else 8000
opt = torch.optim.AdamW(wm.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=WM_STEPS, pct_start=0.05)
W2 = torch.from_numpy(window_starts(data["train_eps"], 2)).to(DEVICE)
offs = torch.arange(2 * K, device=DEVICE)
t0 = time.time(); log = []
for step in range(WM_STEPS):
    i = W2[torch.randint(0, len(W2), (128,), device=DEVICE)]
    blocks = ACT[i[:, None] + offs].view(-1, 2, K, 2)                  # two blocks of 8 actions
    p1, p2 = imagine(wm, Z[i], blocks)
    loss_features = F.mse_loss(p1, Z[i + K]) + F.mse_loss(p2, Z[i + 2 * K])
    loss_reward = F.binary_cross_entropy(wm.coverage(Z[i]), COV[i]) + F.binary_cross_entropy(wm.coverage(p2.detach()), COV[i + 2 * K])
    loss = loss_features + loss_reward
    opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(wm.parameters(), 1.0); opt.step(); sched.step()
    log.append((loss_features.item(), loss_reward.item()))
    if step % 2000 == 0 or step == WM_STEPS - 1:
        rate = (step + 1) / (time.time() - t0)
        print(f"step {step:>6} · features {loss_features.item():.3f} · reward {loss_reward.item():.3f} · ~{(WM_STEPS - step) / rate / 60:.1f} min left")
wm.eval(); torch.save(dict(wm=wm.state_dict(), mu=MU, sd=SD), OUT / "c3_world_model.pt")

---
## 4 · Is the imagination any good? 📏
On **test** episodes: imagine 1–4 blocks ahead (0.8–3.2 s) with the *recorded* actions, and compare with persistence.

In [ ]:
quiz("persistence")

In [ ]:
rows = []
with torch.no_grad():
    for n in [1, 2, 3, 4]:
        i = torch.from_numpy(window_starts(data["test_eps"], n)).to(DEVICE)
        blocks = ACT[i[:, None] + torch.arange(n * K, device=DEVICE)].view(-1, n, K, 2)
        pred = imagine(wm, Z[i], blocks)[-1]
        target = Z[i + n * K]
        rows.append(dict(seconds=n * K / 10,
                         feat_model=F.mse_loss(pred, target).item(), feat_persist=F.mse_loss(Z[i], target).item(),
                         cov_model=(wm.coverage(pred) - COV[i + n * K]).abs().mean().item(),
                         cov_persist=(wm.coverage(Z[i]) - COV[i + n * K]).abs().mean().item()))
print(" ahead | feature error: model vs persistence | coverage error: model vs persistence")
for r in rows:
    print(f"{r['seconds']:5.1f}s | {r['feat_model']:14.3f} vs {r['feat_persist']:6.3f}       | {r['cov_model']:10.3f} vs {r['cov_persist']:.3f}")
fig, axs = plt.subplots(1, 2, figsize=(9, 3))
sec = [r["seconds"] for r in rows]
axs[0].plot(sec, [r["feat_model"] for r in rows], "o-", label="world model"); axs[0].plot(sec, [r["feat_persist"] for r in rows], "o--", label="persistence")
axs[1].plot(sec, [r["cov_model"] for r in rows], "o-"); axs[1].plot(sec, [r["cov_persist"] for r in rows], "o--")
axs[0].set(title="future feature error", xlabel="seconds imagined ahead"); axs[1].set(title="future coverage error", xlabel="seconds imagined ahead"); axs[0].legend(); plt.tight_layout(); plt.show()

## 5 · 👁️ Imagination viewer
For a test demonstration we imagine 4 blocks (3.2 s) ahead using the recorded actions, then show the **training frame whose features are closest** to each imagined state.

### 🧩 Challenge 3 · Find the closest real frame

In [ ]:
def nearest_frames(query, bank):
    return ___          # 🧩 index of the closest bank row for each query

nearest_frames = check("nearest", nearest_frames)

bank_ids = torch.from_numpy(train_frames[::2]).to(DEVICE)
bank = Z[bank_ids].flatten(1)
e = data["test_eps"][1]; i0 = starts[e] + 10
with torch.no_grad():
    blocks = ACT[i0 + torch.arange(4 * K, device=DEVICE)].view(1, 4, K, 2)
    preds = imagine(wm, Z[i0:i0 + 1], blocks)
    idx = [int(bank_ids[nearest_frames(p.flatten(1), bank)[0]]) for p in preds]
fig, axs = plt.subplots(2, 5, figsize=(11, 4.6))
axs[0, 0].imshow(IMGS[i0]); axs[0, 0].set_title("now"); axs[1, 0].axis("off")
for k in range(4):
    axs[0, k + 1].imshow(IMGS[i0 + (k + 1) * K]); axs[0, k + 1].set_title(f"real +{(k + 1) * 0.8:.1f}s")
    axs[1, k + 1].imshow(IMGS[idx[k]]); axs[1, k + 1].set_title(f"imagined +{(k + 1) * 0.8:.1f}s")
for ax in axs.flat: ax.axis("off")
plt.suptitle("top: what happened · bottom: nearest training frame to the world model's imagination"); plt.tight_layout(); plt.show()

<details><summary>🤔 <b>Need a hint?</b></summary>

Pairwise distances, then the smallest per row.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return torch.cdist(query, bank).argmin(dim=1)          # 🧩 index of the closest bank row for each query</pre>

</details>

---
## 6 · The experiment: policy proposes, world model chooses 🧪
**Proposer:** C1's flow-matching state policy (loaded from `c1_flow_policy.pt`; if it's missing we quickly train one).
**Decision rule at every re-plan:** sample N chunks → imagine 16 steps for each → execute the chunk with the highest imagined coverage.

| Condition | What it tests |
|---|---|
| N = 1 | the policy alone |
| N = 8, **random pick** | does merely sampling differently change anything? (control) |
| N = 8 / 32, **world-model pick** | does learned imagination help? |
| N = 8, **perfect pick** | the simulator itself looks ahead: an upper bound on what any verifier could do |

In [ ]:
proposer = load_or_train_c1_policy(data)

def propose(prev_state, state, n):
    return sample_state_chunks(proposer, prev_state, state, n=n)       # (n, 16, 2) in world units

### 🧩 Challenge 4 · Let imagination choose

In [ ]:
quiz("verifier")

In [ ]:
def pick_best(predicted_coverage):
    return ___       # 🧩 the candidate with the best imagined outcome

pick_best = check("pick", pick_best)

def snapshot(env):
    u = env.unwrapped
    return (tuple(u.agent.position), tuple(u.agent.velocity), tuple(u.block.position), float(u.block.angle), tuple(u.block.velocity), float(u.block.angular_velocity))
def restore(env, s):
    u = env.unwrapped
    u.agent.position, u.agent.velocity = s[0], s[1]
    u.block.angle, u.block.position = s[3], s[2]          # angle first (see C1's bug note)
    u.block.velocity, u.block.angular_velocity = s[4], s[5]

scratch = make_env("state"); scratch.reset(seed=0)
def make_selector(n, mode):
    rng = np.random.default_rng(0)
    @torch.no_grad()
    def choose(history, env):
        prev, now = history                                             # last two states
        cands = propose(prev, now, n)
        if n == 1:
            k = 0
        elif mode == "random":
            k = int(rng.integers(n))
        elif mode == "world model":
            z = ((dino_tokens(env.unwrapped._render()[None]).float() - MU.to(DEVICE)) / SD.to(DEVICE)).repeat(n, 1, 1)
            a = torch.tensor(to_unit(cands), device=DEVICE)
            future = imagine(wm, z, a.view(n, 2, K, 2))[-1]
            k = pick_best(wm.coverage(future))
        else:                                                           # perfect: the real simulator looks ahead
            snap, outcome = snapshot(env), []
            for c in cands:
                restore(scratch, snap); best = 0.0
                for act in c:
                    _, _, _, _, info = scratch.step(act); best = max(best, info["coverage"])
                outcome.append(best)
            k = pick_best(torch.tensor(outcome))
        return cands[k]
    return choose

conditions = [(1, "policy alone"), (8, "random"), (8, "world model"), (32, "world model"), (8, "perfect")]
seeds = EVAL_SEEDS[:3] if FAST_DEV_RUN else EVAL_SEEDS
experiment = {}
for n, mode in conditions:
    experiment[f"N={n} · {mode}"] = evaluate(make_selector(n, mode), seeds=seeds, label=f"N={n:>2} · {mode}")

<details><summary>🤔 <b>Need a hint?</b></summary>

<code>torch.argmax</code>, converted to a Python int.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return int(torch.argmax(predicted_coverage))       # 🧩 the candidate with the best imagined outcome</pre>

</details>

### Read the result like a researcher 🔍

In [ ]:
names = list(experiment); means = [experiment[k]["score"] for k in names]
err = [[m - experiment[k]["ci95"][0] for k, m in zip(names, means)], [experiment[k]["ci95"][1] - m for k, m in zip(names, means)]]
plt.figure(figsize=(7, 3)); plt.barh(names, means, xerr=err, capsize=4, color=["#999", "#bbb", "#3b8ea5", "#2a6f80", "#e0a100"])
plt.xlim(0, 1); plt.xlabel("score on 50 scenes (95% CI)"); plt.title("does imagination help choose?"); plt.gca().invert_yaxis(); plt.show()

base = np.array(experiment["N=1 · policy alone"]["per_episode_scores"])
for k in names[1:]:
    diff = np.array(experiment[k]["per_episode_scores"]) - base
    m, lo, hi = bootstrap_ci(diff)
    verdict = "helps" if lo > 0 else ("hurts" if hi < 0 else "no clear difference")
    print(f"{k:>22} − policy alone: {m:+.3f} (95% CI {lo:+.3f} to {hi:+.3f}) → {verdict}")

**How to interpret your table** (write these answers in your report):
1. Is *world model* better than *random pick* at the same N? If not, the model isn't adding information.
2. What fraction of the gap between *policy alone* and *perfect* does the world model close?
3. Does N=32 beat N=8? If it's worse, you may be watching the optimiser's curse: more candidates, more chances to find one that fools the reward head.

In [ ]:
wm_pick = dict(experiment["N=8 · world model"]); wm_pick["label"] = "C3 policy + world-model pick (N=8)"
save_results("c3_results.json", dict(stage="C3", prediction=rows, experiment={k: {kk: vv for kk, vv in v.items()} for k, v in experiment.items()},
                                     wm_pick=wm_pick, wm_steps=WM_STEPS))
print("📄 Draft résumé bullet:")
wm8, pol = experiment["N=8 · world model"]["score"], experiment["N=1 · policy alone"]["score"]
print(f"  • Trained a 3-layer transformer world model in frozen DINOv2 feature space with a learned reward head; used it to rerank"
      f" flow-policy proposals (best-of-8: {wm8:.2f} vs policy alone {pol:.2f}; perfect-verifier bound {experiment['N=8 · perfect']['score']:.2f}).")

### ✅ Stage checklist
- [ ] `c3_world_model.pt`, `c3_results.json` saved
- [ ] The imagination-viewer figure in your report, with one sentence on where imagination drifts
- [ ] A paragraph answering the three interpretation questions

### 🚀 Stretch ideas
* **Plan without a policy:** use CEM (lab 01) over action sequences, scored by imagined coverage. Compare with best-of-N.
* **Uncertainty-aware verifier:** train 3 reward heads and subtract their disagreement (lab 03's cautious ensemble).
* **Longer imagination:** train with 4-block rollouts and re-run section 4.

**Next:** `C4_Finetune_SmolVLA_LeRobot.ipynb`. Put a real 450 M-parameter VLA through the same evaluation.

In [ ]:
progress_report()